# Prepare the ERA5 file for the downscaling notebook

Cuts hourly ERA5 convective and large-scale rain for Typhoon Nando (20 to 24 Sep 2025) over northern Luzon
from Google's public ARCO-ERA5 copy (no account needed) and saves it as `nando_era5.nc` (a few MB).
Run once, then commit the file as `data/nando_era5.nc`. The participant notebook downloads it from there.

In [1]:
%pip install -q gcsfs zarr
import xarray as xr

ds = xr.open_zarr("gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3",
                  chunks=None, storage_options=dict(token="anon"))
print("ERA5 available until:", ds.attrs.get("valid_time_stop"))

ERA5 available until: 2026-06-30


In [2]:
sub = (ds[["convective_precipitation", "large_scale_precipitation"]]
       .sel(time=slice("2025-09-20T00", "2025-09-24T00"), latitude=slice(26, 8), longitude=slice(112, 130))
       .rename(convective_precipitation="cp", large_scale_precipitation="lsp", latitude="lat", longitude="lon")
       .load())
assert sub.sizes["time"] == 97 and int(sub.isnull().sum().to_array().sum()) == 0
enc = {v: {"zlib": True, "complevel": 4, "dtype": "float32"} for v in sub.data_vars}
sub.to_netcdf("nando_era5.nc", encoding=enc)
print(sub)
print("Max hourly rain (mm):", float((sub.cp + sub.lsp).max()) * 1000)

<xarray.Dataset> Size: 4MB
Dimensions:  (time: 97, lat: 73, lon: 73)
Coordinates:
  * time     (time) datetime64[ns] 776B 2025-09-20 ... 2025-09-24
  * lat      (lat) float32 292B 26.0 25.75 25.5 25.25 25.0 ... 8.75 8.5 8.25 8.0
  * lon      (lon) float32 292B 112.0 112.2 112.5 112.8 ... 129.5 129.8 130.0
Data variables:
    cp       (time, lat, lon) float32 2MB 0.0 0.0 0.0 ... 0.0002255 0.0004163
    lsp      (time, lat, lon) float32 2MB 1.978e-05 2.238e-05 ... 0.0001249
Attributes:
    last_updated:           2026-10-25 05:15:22.333497+00:00
    valid_time_start:       1940-01-01
    valid_time_stop:        2026-06-30
    valid_time_stop_era5t:  2026-10-19
Max hourly rain (mm): 34.448400139808655


In [3]:
from google.colab import files
files.download("nando_era5.nc")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>